<a href="https://colab.research.google.com/github/njw257/bigdata/blob/main/week08_5685599_%EB%82%A8%EC%A2%85%EC%9A%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8주차 과제
빅데이터프로그래밍 · 통계학과

**이름:남종욱**
**학번:5685599**

## 과제 내용
필터 개수와 풀링 크기를 변경해 결과를 비교하고, 모델 구조 그림을 작성합니다.

## 제출 방법
모든 셀을 실행해 출력과 그래프가 보이는 상태로 저장한 뒤 `week08_학번_이름.ipynb` 로 제출합니다.

**런타임 > 런타임 유형 변경 > T4 GPU** 를 먼저 선택하세요.


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import pandas as pd

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

CLASSES = ["티셔츠", "바지", "풀오버", "드레스", "코트",
           "샌들", "셔츠", "운동화", "가방", "앵클부츠"]

transform = transforms.ToTensor()
train_set = datasets.FashionMNIST("./data", train=True,  download=True, transform=transform)
test_set  = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False)


device: cuda


100%|██████████| 26.4M/26.4M [00:01<00:00, 13.5MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 199kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.70MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 25.9MB/s]


## 공통 함수
아래 두 함수를 그대로 쓰고 모델만 바꿔 비교하세요.


In [2]:
loss_fn = nn.CrossEntropyLoss()

def evaluate(model):
    model.eval()
    loss_sum = correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += loss_fn(out, y).item() * y.numel()
            correct += (out.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return loss_sum / total, correct / total


def train(model, epochs=6, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = []
    for epoch in range(1, epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
        hist.append(evaluate(model))
        print(f"  epoch {epoch}  acc {hist[-1][1]:.4f}")
    return model, hist


def flat_size(features, size=28):
    with torch.no_grad():
        return features(torch.zeros(1, 1, size, size)).numel()


## 문제 1. 필터 개수 비교 (30점)
필터 개수를 아래 세 가지로 바꿔 각각 학습하고 시험 정확도를 비교하세요.

- (8, 16)
- (16, 32)
- (32, 64)

구조는 `Conv → ReLU → MaxPool` 두 번 + `Flatten → Linear(128) → Linear(10)` 로 통일합니다.
`flat_size` 함수를 써서 Linear 입력 크기를 구하세요.


In [3]:
# 답안

# 필터 개수에 따른 CNN 모델 정의
class FilterCNN(nn.Module):
    def __init__(self, filters, pool_size=2):
        super().__init__()

        # 합성곱 및 풀링 층 구성
        self.features = nn.Sequential(
            nn.Conv2d(1, filters[0], 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(pool_size),
            nn.Conv2d(filters[0], filters[1], 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(pool_size)
        )

        # Flatten 크기 계산 및 분류 층 구성
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size(self.features), 128),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# 필터 개수별 모델 학습
models_filter = {}
hist_filter = {}

for filters in [(8, 16), (16, 32), (32, 64)]:
    print(f"\n필터 {filters}")
    model = FilterCNN(filters)
    models_filter[filters], hist_filter[filters] = train(model)

# 필터 개수별 최종 시험 정확도 출력
for filters, hist in hist_filter.items():
    print(f"필터 {filters} 시험 정확도: {hist[-1][1]:.4f}")



필터 (8, 16)
  epoch 1  acc 0.8488
  epoch 2  acc 0.8593
  epoch 3  acc 0.8699
  epoch 4  acc 0.8856
  epoch 5  acc 0.8832
  epoch 6  acc 0.8868

필터 (16, 32)
  epoch 1  acc 0.8377
  epoch 2  acc 0.8757
  epoch 3  acc 0.8862
  epoch 4  acc 0.8944
  epoch 5  acc 0.9024
  epoch 6  acc 0.8910

필터 (32, 64)
  epoch 1  acc 0.8684
  epoch 2  acc 0.8829
  epoch 3  acc 0.8934
  epoch 4  acc 0.9031
  epoch 5  acc 0.9074
  epoch 6  acc 0.9071
필터 (8, 16) 시험 정확도: 0.8868
필터 (16, 32) 시험 정확도: 0.8910
필터 (32, 64) 시험 정확도: 0.9071


## 문제 2. 풀링 크기 비교 (25점)
필터를 (16, 32)로 고정하고 `MaxPool2d` 의 크기를 **2와 4**로 바꿔 비교하세요.
풀링을 크게 하면 Flatten 크기와 파라미터 수가 어떻게 변하는지도 함께 적으세요.


In [4]:
# 답안

# 풀링 크기별 모델 학습
models_pool = {}
hist_pool = {}

for pool_size in [2, 4]:
    print(f"\n풀링 크기: {pool_size}")
    model = FilterCNN((16, 32), pool_size)
    models_pool[pool_size], hist_pool[pool_size] = train(model)

# 풀링 크기별 결과 출력
for pool_size, model in models_pool.items():
    print(f"\n풀링 크기: {pool_size}")
    print("Flatten 크기:", model.classifier[1].in_features)
    print("파라미터 수:", sum(p.numel() for p in model.parameters()))
    print("시험 정확도:", hist_pool[pool_size][-1][1])



풀링 크기: 2
  epoch 1  acc 0.8609
  epoch 2  acc 0.8711
  epoch 3  acc 0.8890
  epoch 4  acc 0.8936
  epoch 5  acc 0.8987
  epoch 6  acc 0.9034

풀링 크기: 4
  epoch 1  acc 0.7624
  epoch 2  acc 0.8009
  epoch 3  acc 0.8194
  epoch 4  acc 0.8234
  epoch 5  acc 0.8268
  epoch 6  acc 0.8309

풀링 크기: 2
Flatten 크기: 1568
파라미터 수: 206922
시험 정확도: 0.9034

풀링 크기: 4
Flatten 크기: 32
파라미터 수: 10314
시험 정확도: 0.8309


풀링 크기가 2에서 4로 증가하면서 Flatten 크기는 1,568에서 32로, 파라미터 수는 206,922개에서 10,314개로 감소하였다. 시험 정확도는 90.34%에서 83.09%로 감소하였으며, 이는 과도한 풀링으로 이미지의 특징 정보가 손실되었기 때문으로 볼 수 있다.

## 문제 3. 결과 비교표 (20점)
문제 1·2의 결과를 하나의 표로 정리하세요.

| 설정 | 필터 | 풀링 | Flatten 크기 | 파라미터 수 | 시험 정확도 |


In [5]:
# 답안

# 필터 개수별 결과 정리
rows = []

for filters, model in models_filter.items():
    rows.append({
        "설정": "필터 비교",
        "필터": filters,
        "풀링": 2,
        "Flatten 크기": model.classifier[1].in_features,
        "파라미터 수": sum(p.numel() for p in model.parameters()),
        "시험 정확도": hist_filter[filters][-1][1]
    })

# 풀링 크기별 결과 정리
for pool_size, model in models_pool.items():
    rows.append({
        "설정": "풀링 비교",
        "필터": (16, 32),
        "풀링": pool_size,
        "Flatten 크기": model.classifier[1].in_features,
        "파라미터 수": sum(p.numel() for p in model.parameters()),
        "시험 정확도": hist_pool[pool_size][-1][1]
    })

# 결과 비교표 출력
pd.DataFrame(rows)


,설정,필터,풀링,Flatten 크기,파라미터 수,시험 정확도
0,필터 비교,"(8, 16)",2,784,103018,0.8868
1,필터 비교,"(16, 32)",2,1568,206922,0.8910
2,필터 비교,"(32, 64)",2,3136,421642,0.9071
3,풀링 비교,"(16, 32)",2,1568,206922,0.9034
4,풀링 비교,"(16, 32)",4,32,10314,0.8309


## 문제 4. 모델 구조 그림 (15점)
가장 좋았던 모델의 구조를 아래 형식으로 적으세요. 각 단계의 shape을 코드로 출력해 확인한 값을 쓰세요.

```
입력        (batch, 1, 28, 28)
Conv2d      (batch, ?, ?, ?)
ReLU        ...
MaxPool2d   ...
Conv2d      ...
ReLU        ...
MaxPool2d   ...
Flatten     ...
Linear      ...
Linear      (batch, 10)
```


In [8]:
# 단계별 shape 출력 코드

# 가장 정확도가 높은 모델 선택
model = models_filter[(32, 64)]

# 임의 입력 데이터 4개 생성
x = torch.randn(4, 1, 28, 28).to(device)
print("입력:", tuple(x.shape))

# 각 층을 통과하며 shape 출력
model.eval()

with torch.no_grad():
    h = x

    for layer in list(model.features) + list(model.classifier):
        h = layer(h)
        print(f"{type(layer).__name__:12s}: {tuple(h.shape)}")


입력: (4, 1, 28, 28)
Conv2d      : (4, 32, 28, 28)
ReLU        : (4, 32, 28, 28)
MaxPool2d   : (4, 32, 14, 14)
Conv2d      : (4, 64, 14, 14)
ReLU        : (4, 64, 14, 14)
MaxPool2d   : (4, 64, 7, 7)
Flatten     : (4, 3136)
Linear      : (4, 128)
Linear      : (4, 10)


**구조 그림:**

```
(
입력        (batch, 1, 28, 28)
Conv2d      (batch, 32, 28, 28)
ReLU        (batch, 32, 28, 28)
MaxPool2d   (batch, 32, 14, 14)
Conv2d      (batch, 64, 14, 14)
ReLU        (batch, 64, 14, 14)
MaxPool2d   (batch, 64, 7, 7)
Flatten     (batch, 3136)
Linear      (batch, 128)
Linear      (batch, 10)
)
```


## 문제 5. 해석 (10점)
아래 세 가지를 각각 두세 줄로 적으세요.

1. 필터 개수를 늘리면 정확도가 계속 좋아지는가
2. 풀링 크기를 키우면 무엇을 얻고 무엇을 잃는가
3. 7주차 MLP와 비교해 CNN이 나은 이유


**답:**

1. 필터 개수를 늘리면서 시험 정확도가 88.68%, 89.10%, 90.71%로 향상되었다. 필터가 많아지면 다양한 특징을 추출할 수 있지만, 파라미터 수와 연산량도 증가하므로 정확도가 항상 향상되는 것은 아니다.

2. 풀링 크기를 키우면 특징 맵의 크기와 파라미터 수가 감소하여 연산량을 줄일 수 있다. 그러나 이미지의 세부 특징이 손실될 수 있으며, 실제 시험 정확도도 90.34%에서 83.09%로 감소하였다.

3. MLP는 이미지를 1차원으로 펼쳐서 처리하기 때문에 픽셀 간의 공간적 관계를 활용하기 어렵다. 반면 CNN은 합성곱을 통해 이미지의 위치 관계와 특징을 유지하면서 학습하고, 풀링을 통해 위치 변화에도 덜 민감하게 반응할 수 있다.
